# Практична робота №4
## Розділ 8. Екстрагування ключових слів з відібраних за запитом документів
**Варіант:** 8

**Завдання:** Виконати аналітичну обробку текстових документів, сформувати частотний словник, вилучити стоп-слова та визначити найбільш вагомі ключові слова. Реалізувати розрахунок ваги слів базовим частотним методом та методом TF-IDF.

In [1]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

json_data = [
    '"title": "Smartphones laptops and oddball gadgets latest innovations" "source"',
    '"textBody": "Samsung is showcasing the familiar mix of innovative and wacky products"',
    '"title": "Samsung Galaxy phone and watch" "source"',
    '"textBody": "Apple and Samsung compete heavily in the global mobile phone market"'
]

t = " ".join(json_data)

titles = re.findall('"title": "(.+?)" "source"', t)
t = " ".join(titles) + " " + re.sub('"textBody":', ' ', " ".join(json_data))
t = re.sub(r'[-"%\[\]\/?0-9",.\(\)\$+*:;]', ' ', t)
t = t.upper()
t = re.sub(r'\s\w\s', ' ', t)
t = re.sub(r'\s\w\w\s', ' ', t)
t = re.sub(r'\s\s+', ' ', t)

word = t.split()
word.sort()

d = {}
old = ""
n = 0
for w in word:
    if w == old:
        n += 1
    else:
        if old != "":
            d[old] = n
        old = w
        n = 1
if old != "":
    d[old] = n

sorted_keys = sorted(d, key=d.get, reverse=True)
sorted_dict = {w: d[w] for w in sorted_keys}

stop = ["AND", "THE", "IS", "IN", "OF", "ON", "TO", "A"]

M = 10
j = 1
print("ЧАСТОТНИЙ МЕТОД (Завдання за методичкою):")
for w in sorted_keys:
    if w not in stop:
        print(f"{j}. {w}: {sorted_dict[w]}")
        j += 1
    if j > M:
        break

print("\nМЕТОД TF-IDF (Завдання на самостійну роботу):")
docs = [
    "Smartphones laptops and oddball gadgets latest innovations",
    "Samsung is showcasing the familiar mix of innovative and wacky products",
    "Samsung Galaxy phone and watch",
    "Apple and Samsung compete heavily in the global mobile phone market"
]
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(docs)
scores = zip(vectorizer.get_feature_names_out(), np.asarray(tfidf_matrix.sum(axis=0)).ravel())
sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)

for idx, (w, s) in enumerate(sorted_scores[:10], 1):
    print(f"{idx}. {w.upper()}: {s:.4f}")

ЧАСТОТНИЙ МЕТОД (Завдання за методичкою):
1. SAMSUNG: 4
2. PHONE: 3
3. GADGETS: 2
4. GALAXY: 2
5. INNOVATIONS: 2
6. LAPTOPS: 2
7. LATEST: 2
8. ODDBALL: 2
9. SMARTPHONES: 2
10. SOURCE: 2

МЕТОД TF-IDF (Завдання на самостійну роботу):
1. SAMSUNG: 0.8597
2. PHONE: 0.7504
3. GALAXY: 0.5746
4. WATCH: 0.5746
5. GADGETS: 0.4082
6. INNOVATIONS: 0.4082
7. LAPTOPS: 0.4082
8. LATEST: 0.4082
9. ODDBALL: 0.4082
10. SMARTPHONES: 0.4082


### Висновки та відповіді на контрольні запитання

1. **Як відбувається вибір найбільш вагомих слів за заданою тематикою?**
   Відбувається лексичний аналіз масиву документів: об'єднується вміст полів, текст очищується від розділових знаків, цифр та розбивається на окремі слова. Далі рахується абсолютна частота появи кожного слова. З отриманого списку обов'язково видаляються «стоп-слова» (прийменники, сполучники, загальновживані слова), які не несуть змісту. Слова з найбільшою частотою, що залишилися, вважаються найвагомішими.

2. **Яка подальша мета використання отриманих слів по заданій тематиці?**
   Отримані найвагоміші ключові слова стають інформаційними концептами (основними поняттями тексту). На їх основі будується матриця суміжності (фіксується їх спільна поява в реченнях), яка далі використовується для формування та візуалізації графових структур (семантичних мереж) у програмах типу Gephi.

3. **У чому полягає суть іншого методу розрахунку ваги слів, зокрема методу TF-IDF?**
   Метод TF-IDF оцінює слово не лише за його частотою в одному тексті, а й за його унікальністю серед усіх доступних документів. Слово отримує високу вагу, якщо воно часто зустрічається в конкретному документі, але дуже рідко в інших. Це ефективно відфільтровує загальновживану фонову лексику.

4. **У чому полягає суть іншого методу розрахунку ваги слів, зокрема дисперсійного методу?**
   Дисперсійний метод аналізує просторовий розподіл слова в тексті (відстань між сусідніми появами слова). Якщо слово групується в певних смислових абзацах, відстані будуть мати високу дисперсію (нерівномірність), що свідчить про високу значущість слова. Незначущі слова розподілені по тексту рівномірно і мають низьку дисперсію.